# Monthly data for 1 HUC8 basin

This Google Colab notebook uses **Google Earth Engine** and **geemap** to:

1. Find the HUC8 watershed associated with the 1 huc8 basin .
2. Load daily GRIDMET precipitation (`IDAHO_EPSCOR/GRIDMET`).
3. Sum daily precipitation to monthly totals.
4. Calculate the basin-wide mean monthly precipitation.
5. Save the results to a CSV file.

> GRIDMET precipitation variable: `pr`, daily precipitation in **mm/day**. Monthly totals are therefore in **mm/month** after summing daily images.


## 1. Install and import packages

Run this cell first in Colab. You may be asked to restart the runtime after installation; if so, rerun the import/authentication cells after restarting.

In [ ]:
# !pip install -q geemap earthengine-api pandas

In [1]:
import ee
import geemap
import pandas as pd
# from google.colab import files

## 2. Authenticate and initialize Earth Engine

Run the cell and follow the authentication prompts.

In [2]:
try:
    ee.Initialize(project='openet')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='openet')

print('Earth Engine initialized')

Earth Engine initialized


In [3]:
# ee.Authenticate(force=True)
# ee.Initialize(project='openet')

## 3. Set analysis parameters

Adjust `START_YEAR` and `END_YEAR` as needed.

The notebook finds the HUC8 by selecting the watershed polygon that contains a representative point near the Pajaro River / Watsonville area. This avoids relying on a hard-coded HUC8 ID. The selected HUC8 name and code are printed below so you can verify it.

In [4]:
# Analysis period: inclusive years
START_YEAR = 2023
END_YEAR = 2024

# Representative point near the lower Pajaro River / Watsonville, CA.
# Coordinates are longitude, latitude.
PAJARO_POINT = ee.Geometry.Point([-121.75, 36.90])

# Output file
OUT_CSV = 'huc8_monthly_precip_mean.csv'

print(f'Analysis period: {START_YEAR}-{END_YEAR}')

Analysis period: 2023-2024


## 4. Load the HUC8 basin

Dataset: `USGS/WBD/2017/HUC08`, Watershed Boundary Dataset HUC8 polygons.

In [5]:
huc8 = ee.FeatureCollection('USGS/WBD/2017/HUC08')

pajaro_huc8 = huc8.filterBounds(PAJARO_POINT).first()
pajaro_fc = ee.FeatureCollection([pajaro_huc8])
basin_geom = pajaro_huc8.geometry()

# Print metadata for verification.
props = pajaro_huc8.toDictionary().getInfo()
print('Selected HUC8 properties:')
for key in ['huc8', 'name', 'states', 'areasqkm']:
    if key in props:
        print(f'  {key}: {props[key]}')

Selected HUC8 properties:
  huc8: 18060002
  name: Pajaro
  states: CA
  areasqkm: 3368.62


## 5. Map the selected basin

In [6]:
Map = geemap.Map()
Map


Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [7]:
Map = geemap.Map(center=[36.9, -121.75], zoom=8)
Map.addLayer(pajaro_fc, {'color': 'blue'}, 'Selected Pajaro-area HUC8')
Map.addLayer(PAJARO_POINT, {'color': 'red'}, 'Representative Pajaro point')
Map

EEException: Permission 'earthengine.maps.create' denied on resource 'projects/openet' (or it may not exist).

## 6. Load daily GRIDMET precipitation

GRIDMET precipitation band `pr` is daily precipitation in millimeters. We filter by date and basin bounds.

In [ ]:
start_date = ee.Date.fromYMD(START_YEAR, 1, 1)
end_date = ee.Date.fromYMD(END_YEAR + 1, 1, 1)

gridmet_pr = (
    ee.ImageCollection('IDAHO_EPSCOR/GRIDMET')
    .filterDate(start_date, end_date)
    .filterBounds(basin_geom)
    .select('pr')
)

print('Daily GRIDMET images:', gridmet_pr.size().getInfo())

## 7. Sum daily precipitation to monthly totals

For each month, this creates one image where pixel values are the sum of all daily precipitation values in that month. Each image is tagged with year, month, date, and number of daily images used.

In [ ]:
def monthly_precip_image(year_month):
    year = ee.Number(ee.List(year_month).get(0))
    month = ee.Number(ee.List(year_month).get(1))
    month_start = ee.Date.fromYMD(year, month, 1)
    month_end = month_start.advance(1, 'month')

    monthly_sum = (
        gridmet_pr
        .filterDate(month_start, month_end)
        .sum()
        .rename('monthly_precip_mm')
        .set({
            'year': year,
            'month': month,
            'date': month_start.format('YYYY-MM'),
            'system:time_start': month_start.millis(),
            'daily_image_count': gridmet_pr.filterDate(month_start, month_end).size()
        })
    )
    return monthly_sum

years = ee.List.sequence(START_YEAR, END_YEAR)
months = ee.List.sequence(1, 12)
year_months = years.map(lambda y: months.map(lambda m: ee.List([y, m]))).flatten()

monthly_pr = ee.ImageCollection.fromImages(year_months.map(monthly_precip_image))

print('Monthly images:', monthly_pr.size().getInfo())

## 8. Calculate basin mean monthly precipitation

This reduces each monthly precipitation image over the selected HUC8 geometry and returns one row per month.

`mean_monthly_precip_mm` is the **area-averaged basin mean of monthly total precipitation**, in millimeters.

In [ ]:
def monthly_basin_mean(img):
    mean_dict = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=basin_geom,
        scale=4000,              # GRIDMET native resolution is about 4 km
        bestEffort=True,
        maxPixels=1e13
    )

    return ee.Feature(None, {
        'date': img.get('date'),
        'year': img.get('year'),
        'month': img.get('month'),
        'mean_monthly_precip_mm': mean_dict.get('monthly_precip_mm'),
        'daily_image_count': img.get('daily_image_count')
    })

monthly_stats_fc = ee.FeatureCollection(monthly_pr.map(monthly_basin_mean))

# Preview first few records from Earth Engine.
monthly_stats_fc.limit(5).getInfo()

## 9. Convert results to a pandas DataFrame

In [ ]:
# Pull the FeatureCollection into a pandas DataFrame.
# For long time periods, this is still usually small because there is only one record per month.
df = geemap.ee_to_df(monthly_stats_fc)

# Clean and sort output.
df = df[['date', 'year', 'month', 'mean_monthly_precip_mm', 'daily_image_count']]
df = df.sort_values(['year', 'month']).reset_index(drop=True)

# Optional: round precipitation for cleaner CSV output.
df['mean_monthly_precip_mm'] = df['mean_monthly_precip_mm'].astype(float).round(3)

print(df.head())
print(df.tail())

## 10. Save CSV and download it from Colab

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')

files.download(OUT_CSV)

## Optional: Export the CSV to Google Drive

Use this option if you prefer saving directly to Google Drive instead of downloading from the browser.

In [ ]:
# Optional Google Drive export
# from google.colab import drive
# drive.mount('/content/drive')
# drive_path = f'/content/drive/MyDrive/{OUT_CSV}'
# df.to_csv(drive_path, index=False)
# print(f'Saved to Google Drive: {drive_path}')